In [1]:
import pandas as pd

df_prices = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv')
# Re-load the signal data without setting the index
df_signals = pd.read_csv('E:\Signal Backtesting\Input\Signature_AI_Results_Final.csv')

# Convert 'Datetime' column to datetime
df_signals['Datetime'] = pd.to_datetime(df_signals['Datetime'])
df_prices['Datetime'] = pd.to_datetime(df_prices['Datetime'])

# Set 'Datetime' as index for price dataframe for easy slicing
df_prices.set_index('Datetime', inplace=True)

# Sort the price dataframe by datetime in ascending order
df_prices.sort_index(inplace=True)


In [4]:
import pandas as pd
import plotly.graph_objects as go
from datetime import timedelta

# Define the function to calculate percentage change
def calculate_percentage_change(signal, signal_datetime, df_prices, time_window):
    # Find the nearest previous datetime if the exact one doesn't exist
    if signal_datetime not in df_prices.index:
        signal_datetime = df_prices.index[df_prices.index.get_loc(signal_datetime, method='pad')]
    
    # Get the price at signal datetime
    signal_price = df_prices.loc[signal_datetime, 'Open']
    
    # Define the time window
    end_datetime = signal_datetime + timedelta(minutes=time_window)
    
    # Slice the price dataframe for the time window
    price_window = df_prices.loc[signal_datetime:end_datetime]
    
    # Calculate lowest/highest price in the window based on signal
    if signal == 2:  # Buy signal, find lowest price
        target_price = price_window['Low'].min()
    elif signal == -2:  # Sell signal, find highest price
        target_price = price_window['High'].max()
    
    # Calculate percentage change
    percentage_change = ((target_price - signal_price) / signal_price) * 100
    
    return signal_price, target_price, percentage_change

# Time windows to analyze
time_windows = [120, 180, 240]

# Initialize dictionaries to store the results for each time window
buy_changes_dict = {window: [] for window in time_windows}
sell_changes_dict = {window: [] for window in time_windows}

# Loop through each signal and calculate percentage changes for each time window
for index, row in df_signals.iterrows():
    signal = row['Signal']
    signal_datetime = row['Datetime']
    for window in time_windows:
        _, _, percentage_change = calculate_percentage_change(signal, signal_datetime, df_prices, window)
        if signal == 2:
            buy_changes_dict[window].append(percentage_change)
        elif signal == -2:
            sell_changes_dict[window].append(percentage_change)

# Create traces for each time window for the buy signals
buy_traces = []
for window in time_windows:
    buy_traces.append(go.Histogram(x=buy_changes_dict[window], nbinsx=80, name=f'Buy {window} min'))

# Create traces for each time window for the sell signals
sell_traces = []
for window in time_windows:
    sell_traces.append(go.Histogram(x=sell_changes_dict[window], nbinsx=80, name=f'Sell {window} min'))

# Create the figure for buy signals
fig_buy = go.Figure(data=buy_traces)
fig_buy.update_layout(title='Buy Position Percentage Changes', barmode='overlay', width=1200)
for trace in fig_buy.data:
    trace.opacity = 0.75

# Create the figure for sell signals
fig_sell = go.Figure(data=sell_traces)
fig_sell.update_layout(title='Sell Position Percentage Changes', barmode='overlay', width=1200)
for trace in fig_sell.data:
    trace.opacity = 0.75

# Show the plots
fig_buy.show()
fig_sell.show()
